# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Loading the dataset**

In [1]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [2]:
table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector built from February only** (the prior month), so nothing here can see into
March, where the label lives. Two ratio features (`ctr_feb`, `engagement_rate_feb`) have a
possible zero-denominator, handled with an explicit `had_*` flag rather than silently filling
0 — a page with no impressions and a page with impressions but zero clicks are different
situations, and collapsing them into the same "0" would lose that distinction. Position is
also converted from a raw number into a bucketed, one-hot-encoded tier
(`tier_top_3` / `tier_page_1` / `tier_page_2` / `tier_deep` / `tier_no_position`) so it's
usable as a categorical signal, not just a continuous one. Every fill decision is explained
in Section 2, not left implicit in the code.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

PRIOR_MONTH  = "2026-02"
RECENT_MONTH = "2026-03"

# Raw monthly aggregates, Feb only -- no March data touched here
raw_feb = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position)      AS avg_position_feb,
           SUM(gsc_impressions)       AS impressions_feb,
           SUM(gsc_clicks)            AS clicks_feb,
           SUM(ga4_sessions)          AS ga4_sessions_feb,
           SUM(ga4_engaged_sessions)  AS ga4_engaged_sessions_feb,
           SUM(ga4_pageviews)         AS ga4_pageviews_feb,
           SUM(scroll_events)         AS scroll_events_feb,
           SUM(sessions_ai)           AS sessions_ai_feb
    FROM read_parquet('{table}')
    WHERE month = '{PRIOR_MONTH}'
    GROUP BY content_hash_id
""").df()

feat = raw_feb.copy()

# --- Engineered ratio features, with explicit fill handling ---
# CTR: undefined when impressions_feb == 0, not truly "0" -- track that separately
feat["had_impressions_feb"] = (feat["impressions_feb"] > 0).astype(int)
feat["ctr_feb"] = np.where(feat["impressions_feb"] > 0,
                            feat["clicks_feb"] / feat["impressions_feb"], 0.0)

feat["had_ga4_sessions_feb"] = (feat["ga4_sessions_feb"] > 0).astype(int)
feat["engagement_rate_feb"] = np.where(feat["ga4_sessions_feb"] > 0,
                                        feat["ga4_engaged_sessions_feb"] / feat["ga4_sessions_feb"], 0.0)

feat["scroll_rate_feb"] = np.where(feat["ga4_pageviews_feb"] > 0,
                                    feat["scroll_events_feb"] / feat["ga4_pageviews_feb"], 0.0)

feat["ai_referral_share_feb"] = np.where(feat["ga4_sessions_feb"] > 0,
                                          feat["sessions_ai_feb"] / feat["ga4_sessions_feb"], 0.0)

# --- Categorical handling: bucket avg_position into a tier, then one-hot encode ---
def position_tier(p):
    if pd.isna(p):     return "no_position"
    if p <= 3:          return "top_3"
    if p <= 10:         return "page_1"
    if p <= 20:         return "page_2"
    return "deep"

feat["position_tier_feb"] = feat["avg_position_feb"].apply(position_tier)
tier_dummies = pd.get_dummies(feat["position_tier_feb"], prefix="tier")
feat = pd.concat([feat, tier_dummies], axis=1)

# fill any remaining NaNs (pages with zero rows in some sub-metric)
feat = feat.fillna(0)

print(f"Feature frame: {len(feat)} pages, {feat.shape[1]} columns")
feat.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 321546 pages, 21 columns


,content_hash_id,avg_position_feb,impressions_feb,clicks_feb,ga4_sessions_feb,ga4_engaged_sessions_feb,ga4_pageviews_feb,scroll_events_feb,sessions_ai_feb,had_impressions_feb,...,had_ga4_sessions_feb,engagement_rate_feb,scroll_rate_feb,ai_referral_share_feb,position_tier_feb,tier_deep,tier_no_position,tier_page_1,tier_page_2,tier_top_3
0,content_1eea820697c3b95a,12.946228,299.0,0.0,0.0,0.0,0.0,0.0,0.0,1,...,0,0.0,0.0,0.0,page_2,False,False,False,True,False
1,content_9abd8b303f805847,6.495085,733.0,6.0,6.0,0.0,6.0,0.0,0.0,1,...,1,0.0,0.0,0.0,page_1,False,False,True,False,False
2,content_5f58c55cbfee172a,10.490023,514.0,0.0,1.0,0.0,1.0,0.0,0.0,1,...,1,0.0,0.0,0.0,page_2,False,False,False,True,False


- **`avg_position_feb`** — mean Google search position across February. Missing when a page
  had zero GSC impressions that month; bucketed as `"no_position"` rather than filled with a
  number, since a missing position isn't a real position value. Available before the March
  decision moment — entirely February data.
- **`impressions_feb`** — total GSC impressions, February. No missing values possible (sums
  to 0 if absent). Available before the decision moment.
- **`ctr_feb`** (engineered) — clicks_feb / impressions_feb. Undefined, not zero, when
  impressions_feb is 0 — filled to 0 but flagged with a companion `had_impressions_feb` flag
  so "genuinely zero CTR" and "no exposure at all" aren't silently treated as the same thing.
  Available before the decision moment.
- **`engagement_rate_feb`** (engineered) — ga4_engaged_sessions_feb / ga4_sessions_feb. Same
  zero-denominator issue as CTR, same fix (`had_ga4_sessions_feb` flag). Available before the
  decision moment.
- **`scroll_rate_feb`** (engineered) — scroll_events_feb / ga4_pageviews_feb. Filled to 0 on
  a zero denominator; no separate flag added since pageviews of 0 also means impressions/
  clicks are 0, already captured by the other flags. Available before the decision moment.
- **`ai_referral_share_feb`** (engineered) — sessions_ai_feb / ga4_sessions_feb. Same
  zero-denominator handling as engagement_rate. Available before the decision moment.
- **`position_tier_feb`** (categorical, one-hot encoded → `tier_top_3`, `tier_page_1`,
  `tier_page_2`, `tier_deep`, `tier_no_position`) — a bucketed version of avg_position_feb.
  Encoded as dummies rather than left as text so it can actually be fed to a model. Same
  availability as avg_position_feb.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Feature availability check -- confirm every feature came only from PRIOR_MONTH rows:")
print(f"Prior month used: {PRIOR_MONTH}")
print(f"Recent month used only for the label (Section 3/4), never for features: {RECENT_MONTH}")
print("\nMissing-value handling summary:")
print(f"  had_impressions_feb == 0 for {(feat['had_impressions_feb']==0).sum()} pages (ctr_feb filled to 0 for these)")
print(f"  had_ga4_sessions_feb == 0 for {(feat['had_ga4_sessions_feb']==0).sum()} pages (engagement_rate/ai_referral_share filled to 0)")
print(f"  position_tier_feb == 'no_position' for {(feat['position_tier_feb']=='no_position').sum()} pages")

Feature availability check -- confirm every feature came only from PRIOR_MONTH rows:
Prior month used: 2026-02
Recent month used only for the label (Section 3/4), never for features: 2026-03

Missing-value handling summary:
  had_impressions_feb == 0 for 167987 pages (ctr_feb filled to 0 for these)
  had_ga4_sessions_feb == 0 for 288199 pages (engagement_rate/ai_referral_share filled to 0)
  position_tier_feb == 'no_position' for 167987 pages


Two checks: first, prove every feature column's SQL only ever filtered on `PRIOR_MONTH` —
no query above touches `RECENT_MONTH`. Second, the actual attack: inject a deliberately
label-derived column (`clicks_mar`, straight from the recent month, the same trap as
notebook 02 and the w03 contract) into the honest feature set and watch Precision@50 jump
toward-perfect. That confirms the test itself is sensitive enough to catch a real leak — and
by extension, that the honest feature set above doesn't already contain one.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Label: Feb vs March click comparison, non-overlapping months (same as w03)
clicks = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN month = '{PRIOR_MONTH}'  THEN gsc_clicks END) AS clicks_feb_check,
           SUM(CASE WHEN month = '{RECENT_MONTH}' THEN gsc_clicks END) AS clicks_mar
    FROM read_parquet('{table}')
    WHERE month IN ('{PRIOR_MONTH}', '{RECENT_MONTH}')
    GROUP BY content_hash_id
""").df()

panel = feat.merge(clicks, on="content_hash_id", how="inner").fillna(0)
panel["is_declining"] = (panel["clicks_mar"] < panel["clicks_feb_check"]).astype(int)
y = panel["is_declining"].values

honest_cols = ["avg_position_feb", "impressions_feb", "ctr_feb", "engagement_rate_feb",
               "scroll_rate_feb", "ai_referral_share_feb",
               "tier_top_3", "tier_page_1", "tier_page_2", "tier_deep", "tier_no_position"]

X_honest = panel[honest_cols]
t_honest = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
p50_honest = precision_at_k(t_honest.predict_proba(X_honest)[:, 1], y, 50)
print(f"HONEST feature set  Precision@50: {p50_honest:.3f}")

# THE ATTACK: inject a label-derived column on purpose
X_leaky = panel[honest_cols + ["clicks_mar"]]
t_leaky = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
p50_leaky = precision_at_k(t_leaky.predict_proba(X_leaky)[:, 1], y, 50)
print(f"LEAKY  feature set   Precision@50: {p50_leaky:.3f}  <- jumped because clicks_mar is IN the label")

print(f"\nJump: {p50_leaky - p50_honest:+.3f}. The honest {p50_honest:.3f} is the number that goes in the contract.")
del X_leaky, t_leaky  # drop the leaky artifact so it can't accidentally get reused downstream

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

HONEST feature set  Precision@50: 0.660
LEAKY  feature set   Precision@50: 1.000  <- jumped because clicks_mar is IN the label

Jump: +0.340. The honest 0.660 is the number that goes in the contract.


- **`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
  `ai_other`** — the per-platform AI breakdown. Every sample row seen so far reads 0 across
  all seven; including seven mostly-empty columns risks adding noise before checking their
  real non-zero rate. `ai_referral_share_feb` (the aggregate `sessions_ai`) is kept instead,
  as one feature rather than seven sparse ones.
- **`sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`,
  `sessions_paid`** — individual channel session columns. Likely overlap with `ga4_sessions`
  (the total), so summing them alongside `ga4_sessions_feb`-derived ratios risks
  double-counting the same underlying sessions.
- **`gsc_sum_position`** — a raw sum isn't meaningful without dividing by impressions;
  `gsc_avg_position` already does that division, so the sum is redundant.
- **`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available`** —
  coverage/context flags, not signals of decline. Useful for filtering or explaining a score,
  not for feeding into it directly.
- **`report_date`, `month`, `client_hash_id`, `content_hash_id`** — identifiers and time
  keys, kept as context for grouping and the client-holdout split, never as features.
- **`clicks_mar`** — deliberately excluded after Section 3 proved it's label-derived; kept
  out of the file entirely rather than just unused, so it can't get reintroduced by accident
  later.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
final_features = honest_cols
excluded_from_raw = [c for c in con.sql(f"DESCRIBE SELECT * FROM read_parquet('{table}')").df()["column_name"]
                      if c not in final_features + ["report_date","month","client_hash_id","content_hash_id",
                                                      "client_has_gsc","client_has_ga4",
                                                      "gsc_data_available","ga4_data_available",
                                                      "gsc_impressions","gsc_clicks","gsc_avg_position",
                                                      "ga4_sessions","ga4_engaged_sessions","ga4_pageviews",
                                                      "scroll_events","sessions_ai"]]
print(f"Final feature set ({len(final_features)} columns):", final_features)
print(f"\nRaw columns excluded entirely from the feature build: {excluded_from_raw}")


Final feature set (11 columns): ['avg_position_feb', 'impressions_feb', 'ctr_feb', 'engagement_rate_feb', 'scroll_rate_feb', 'ai_referral_share_feb', 'tier_top_3', 'tier_page_1', 'tier_page_2', 'tier_deep', 'tier_no_position']

Raw columns excluded entirely from the feature build: ['gsc_sum_position', 'ga4_users', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.